In [1]:
!pip install -q langchain langchain-groq gspread pydantic

In [2]:
import os
from google.colab import userdata
from langchain_groq import ChatGroq

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.3,
    max_retries=3
)

In [3]:
SYSTEM = """ROLE:
You are VulnAgent, an AI-based vulnerability assessment
and prioritization assistant.

FACTS:
The user provides vulnerability descriptions.
Analyze only the information provided by the user.
Consider vulnerability type, possible impact, severity,
and remediation.

RULES:
Do not claim that you scanned or tested a real system.
Do not invent scan results or CVSS scores.
If information is insufficient, clearly mention it.
Provide defensive security recommendations.
Do not provide instructions to exploit real systems.

FORMAT:
Return a clear vulnerability analysis containing:
Vulnerability Name
Severity
Impact
Remediation
Verification
Confidence

Keep the response simple, structured, and understandable.
"""

In [4]:
from typing import Literal
from pydantic import BaseModel, Field

class VulnerabilityAssessment(BaseModel):

    vulnerability: str = Field(
        description="Name or type of vulnerability"
    )

    severity: Literal[
        "Critical", "High", "Medium", "Low", "Unknown"
    ]

    impact: str

    remediation: str

    verification: str

    confidence: Literal[
        "High", "Medium", "Low"
    ]

In [5]:
from langchain.tools import tool

@tool
def get_vulnerability_guidance(vulnerability: str) -> str:
    """Get defensive security guidance for a vulnerability.
    Use when the user asks about a vulnerability type
    or needs remediation recommendations.
    """

    vuln = vulnerability.lower()

    if "sql injection" in vuln:
        return """
        Use parameterized queries or prepared statements.
        Avoid directly concatenating user input into SQL.
        Apply least-privilege database permissions.
        """

    elif "xss" in vuln or "cross-site scripting" in vuln:
        return """
        Apply context-aware output encoding.
        Sanitize untrusted HTML where appropriate.
        Use a suitable Content Security Policy.
        """

    elif "password" in vuln:
        return """
        Enforce strong password policies.
        Store passwords using a secure password hashing
        algorithm such as Argon2id or bcrypt.
        Enable rate limiting and MFA where appropriate.
        """

    elif "access control" in vuln or "idor" in vuln:
        return """
        Enforce server-side authorization checks.
        Verify ownership and permissions for every request.
        Do not rely only on user-controlled object IDs.
        """

    elif "remote code execution" in vuln or "rce" in vuln:
        return """
        Validate inputs and avoid unsafe command execution.
        Patch vulnerable components.
        Apply least privilege and isolate services.
        """

    else:
        return """
        Identify the affected component.
        Review the root cause and possible impact.
        Apply secure configuration and appropriate patches.
        Verify the fix in an authorized test environment.
        """


@tool
def get_severity_guidance(severity: str) -> str:
    """Explain a vulnerability severity category.
    Use when the user asks what a severity level means.
    This is general guidance, not an official CVSS score.
    """

    severity = severity.lower().strip()

    if severity == "critical":
        return "Critical: potentially severe impact requiring urgent attention."

    elif severity == "high":
        return "High: significant risk that should be prioritized."

    elif severity == "medium":
        return "Medium: meaningful risk that should be addressed."

    elif severity == "low":
        return "Low: limited risk that should still be reviewed."

    else:
        return "Unknown: insufficient information to determine severity."

In [6]:
from google.colab import auth
import gspread
from google.auth import default

auth.authenticate_user()

creds, _ = default()
gc = gspread.authorize(creds)

sheet_name = "VulnAgent Assessments"

try:
    sh = gc.open(sheet_name)
    print("Existing spreadsheet opened.")

except gspread.SpreadsheetNotFound:
    sh = gc.create(sheet_name)
    print("New spreadsheet created.")

ws = sh.sheet1
ws.update_title("Vulnerability Records")

if not ws.row_values(1):
    ws.append_row([
        "Vulnerability",
        "Severity",
        "Impact",
        "Remediation",
        "Verification",
        "Confidence"
    ])

print("Google Sheet ready:")
print(sh.url)

Existing spreadsheet opened.
Google Sheet ready:
https://docs.google.com/spreadsheets/d/1pUT6zMegUjz8tXwbkAfi3EMIp9n1VNX3Jo4c90pqLX4


In [7]:
@tool
def log_vulnerability_assessment(
    vulnerability: str,
    severity: str,
    impact: str,
    remediation: str,
    verification: str,
    confidence: str
) -> str:
    """Save a vulnerability assessment to Google Sheets.
    Use only when the user explicitly asks to save an
    assessment and all required details are available.
    """

    try:
        record = VulnerabilityAssessment(
            vulnerability=vulnerability,
            severity=severity,
            impact=impact,
            remediation=remediation,
            verification=verification,
            confidence=confidence
        )

        ws.append_row([
            record.vulnerability,
            record.severity,
            record.impact,
            record.remediation,
            record.verification,
            record.confidence
        ])

        return "Assessment saved successfully."

    except Exception as error:
        return f"Could not save assessment: {error}"

In [8]:
from langchain.agents import create_agent

DAY4_SYSTEM = SYSTEM + """

TOOLS:
Use get_vulnerability_guidance when defensive guidance
is required.

Use get_severity_guidance when explaining severity.

Use log_vulnerability_assessment only when the user
explicitly asks to save an assessment.

Do not save incomplete records.
Do not invent scan results or CVSS scores.
"""

agent = create_agent(
    model=llm,
    tools=[
        get_vulnerability_guidance,
        get_severity_guidance,
        log_vulnerability_assessment
    ],
    system_prompt=DAY4_SYSTEM
)

print("VulnAgent Day 4 agent created.")

VulnAgent Day 4 agent created.


In [9]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": """
            A login page directly inserts user input into SQL.
            Explain the vulnerability and its remediation.
            """
        }
    ]
})

print(result["messages"][-1].content)

**Vulnerability Name:** SQL Injection in Authentication (Login Page)  

**Severity:** High  

**Impact:**  
- An attacker can supply crafted input (e.g., `' OR '1'='1`) that is concatenated directly into the SQL statement.  
- This may allow bypassing authentication, retrieving or modifying user credentials, and potentially executing arbitrary queries against the database.  
- Consequences include unauthorized access, data leakage, data tampering, and escalation to full system compromise.  

**Remediation:**  
- Replace string concatenation with **parameterized queries** or **prepared statements** in the data‑access layer.  
- Use a database‑access library/ORM that enforces binding of user input as parameters (e.g., `mysqli_stmt_prepare`, `PDO::prepare`, `SqlCommand` with parameters, etc.).  
- Enforce **least‑privilege** database accounts: the account used by the web application should only have the permissions it truly needs (e.g., SELECT on the users table, no DROP/ALTER rights).  


In [35]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Explain the severity of a high-risk vulnerability."
        }
    ]
})

print(result["messages"][-1].content)

**Severity: High**

A high‑severity vulnerability indicates a significant risk to the affected system. It typically means that an attacker could:

- **Achieve substantial impact** such as unauthorized data access, data modification, or service disruption.
- **Exploit the flaw with relatively low effort** (e.g., no authentication required, or easily guessable credentials).
- **Reach a wide attack surface** (e.g., exposed to the internet or reachable by many users).

Because of these factors, high‑severity issues should be **prioritized for remediation** as soon as possible—ideally within a short, defined timeframe (e.g., days rather than weeks). Delaying mitigation increases the likelihood of successful exploitation and potential damage to confidentiality, integrity, or availability.


In [10]:
extractor = llm.with_structured_output(
    VulnerabilityAssessment
)

messy_messages = [
    "login uses sql query with user input, maybe sql injection",
    "website reflects my script in browser, xss issue",
    "users use 123456 as password, weak password problem",
    "changing account id shows other user's data, IDOR",
    "application runs unsafe commands, possible RCE"
]

structured_results = []

for message in messy_messages:

    print("\nInput:", message)

    try:
        record = extractor.invoke(message)

        structured_results.append(record)

        print(record.model_dump())

    except Exception as error:
        print("Structured output error:", error)


Input: login uses sql query with user input, maybe sql injection
{'vulnerability': 'SQL Injection in login authentication query', 'severity': 'Critical', 'impact': 'An attacker could bypass authentication, gain unauthorized access, retrieve or modify sensitive data, and potentially execute arbitrary database commands.', 'remediation': 'Use prepared statements / parameterized queries for all user-supplied input. Validate and sanitize inputs. Implement stored procedures with bound parameters. Employ ORM that abstracts SQL. Enforce least privilege for database accounts.', 'verification': 'Attempt login with payloads such as "\' OR \'1\'=\'1" and observe if authentication is bypassed. Use automated scanners (e.g., sqlmap) to confirm injection points.', 'confidence': 'High'}

Input: website reflects my script in browser, xss issue
{'vulnerability': 'Reflected Cross‑Site Scripting (XSS)', 'severity': 'High', 'impact': "An attacker can inject malicious JavaScript that runs in the victim's br

In [39]:
sample_record = structured_results[0]

save_result = log_vulnerability_assessment.invoke(
    sample_record.model_dump()
)

print(save_result)

Assessment saved successfully.


In [11]:
trace_result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": """
            Explain SQL Injection and its prevention.
            """
        }
    ]
})

for message in trace_result["messages"]:
    message.pretty_print()

================================ Human Message =================================


            Explain SQL Injection and its prevention.
            
================================== Ai Message ==================================
Tool Calls:
  get_vulnerability_guidance (fc_507933c4-84f8-4bf9-9752-865e8bcf9f21)
 Call ID: fc_507933c4-84f8-4bf9-9752-865e8bcf9f21
  Args:
    vulnerability: SQL Injection
================================= Tool Message =================================
Name: get_vulnerability_guidance


        Use parameterized queries or prepared statements.
        Avoid directly concatenating user input into SQL.
        Apply least-privilege database permissions.
        
================================== Ai Message ==================================
Tool Calls:
  get_severity_guidance (fc_bc86fa2d-f1fb-4d0f-9e9c-2942f451f05f)
 Call ID: fc_bc86fa2d-f1fb-4d0f-9e9c-2942f451f05f
  Args:
    severity: High
================================= Tool Message ===================

In [12]:
tool_calls = 0
model_messages = 0

for message in trace_result["messages"]:

    if message.type == "ai":
        model_messages += 1

        tool_calls += len(
            getattr(message, "tool_calls", [])
        )

print("Tool calls:", tool_calls)
print("AI messages:", model_messages)

Tool calls: 2
AI messages: 3


In [13]:
@tool
def tool1(vulnerability: str) -> str:
    """Does stuff."""
    return get_vulnerability_guidance.invoke({
        "vulnerability": vulnerability
    })

In [14]:
test_agent = create_agent(
    model=llm,
    tools=[tool1, get_severity_guidance],
    system_prompt=DAY4_SYSTEM
)

In [15]:
test_result = test_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What defensive steps should I take for SQL Injection?"
        }
    ]
})

for message in test_result["messages"]:
    message.pretty_print()

================================ Human Message =================================

What defensive steps should I take for SQL Injection?
================================== Ai Message ==================================

**Vulnerability Name:** SQL Injection  

**Severity:** High  

**Impact**  
- Unauthorized data disclosure (e.g., reading confidential records).  
- Data manipulation or deletion.  
- Privilege escalation to administrative accounts.  
- Potential full system compromise if the database server can be leveraged for code execution.  

**Remediation**  
1. **Use Parameterized Queries / Prepared Statements**  
   - Bind variables rather than concatenating user input into SQL strings.  
   - Most modern frameworks (e.g., JDBC, .NET, PDO, Django ORM) support this out‑of‑the‑box.  

2. **Employ Stored Procedures**  
   - When properly written (no dynamic SQL inside), stored procedures limit injection vectors.  

3. **Input Validation & Sanitization**  
   - Reject or strictly whit

In [16]:
@tool
def log_vulnerability_assessment(
    vulnerability: str,
    severity: str,
    impact: str,
    remediation: str,
    verification: str,
    confidence: str
) -> str:
    """Save a vulnerability assessment to Google Sheets.
    Avoid adding an identical assessment more than once."""

    new_record = {
        "Vulnerability": vulnerability.strip(),
        "Severity": severity.strip(),
        "Impact": impact.strip(),
        "Remediation": remediation.strip(),
        "Verification": verification.strip(),
        "Confidence": confidence.strip()
    }

    # Read existing records from the sheet
    existing_records = ws.get_all_records()

    # Check whether this exact assessment already exists
    for record in existing_records:
        if all(
            str(record.get(column, "")).strip() == value
            for column, value in new_record.items()
        ):
            return "This assessment already exists. Duplicate row skipped."

    # Add only if it is not a duplicate
    ws.append_row(list(new_record.values()))

    return "Vulnerability assessment saved successfully."

In [17]:
agent = create_agent(
    model=llm,
    tools=[
        get_vulnerability_guidance,
        get_severity_guidance,
        log_vulnerability_assessment
    ],
    system_prompt=DAY4_SYSTEM
)

In [18]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "A website stores user passwords in plain text."
        }
    ]
})

for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

A website stores user passwords in plain text.
================================== Ai Message ==================================
Tool Calls:
  get_vulnerability_guidance (fc_9865133c-1efc-4554-8d3e-a1ffb2529525)
 Call ID: fc_9865133c-1efc-4554-8d3e-a1ffb2529525
  Args:
    vulnerability: Plain text password storage
================================= Tool Message =================================
Name: get_vulnerability_guidance


        Enforce strong password policies.
        Store passwords using a secure password hashing
        algorithm such as Argon2id or bcrypt.
        Enable rate limiting and MFA where appropriate.
        
================================== Ai Message ==================================

**Vulnerability Name:** Plain‑text Password Storage  

**Severity:** High  

**Impact:**  
- Anyone with access to the database (e.g., insiders, attackers who breach the server, backup files, log

In [19]:
records = ws.get_all_records()

print("Total records in Google Sheets:", len(records))

for i, record in enumerate(records, start=1):
    print(i, record["Vulnerability"])

Total records in Google Sheets: 5
1 SQL Injection
2 SQL Injection
3 SQL Injection
4 SQL Injection
5 SQL Injection


In [21]:
records = ws.get_all_records()

seen = set()
rows_to_delete = []

for row_number, record in enumerate(records, start=2):
    key = tuple(str(value).strip() for value in record.values())

    if key in seen:
        rows_to_delete.append(row_number)
    else:
        seen.add(key)

for row_number in reversed(rows_to_delete):
    ws.delete_rows(row_number)

print("Duplicate rows deleted:", len(rows_to_delete))
print("Remaining records:", len(ws.get_all_records()))

Duplicate rows deleted: 0
Remaining records: 1


In [23]:
print(ws.get_all_records())

[{'Vulnerability': 'SQL Injection', 'Severity': 'Critical', 'Impact': 'An attacker could bypass authentication, retrieve or modify user data, and potentially gain full system access.', 'Remediation': 'Use prepared statements or parameterized queries for all user-supplied inputs. Validate and sanitize inputs. Implement stored procedures with proper parameter binding. Consider using an ORM that abstracts SQL generation.', 'Verification': 'Attempt login with payloads such as "\' OR \'1\'=\'1" and observe if authentication is bypassed. Check database logs for unexpected query structures.', 'Confidence': 'High'}]


In [24]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": """
            The application allows users to set very weak passwords
            such as 123456 and does not enforce password strength.
            Explain the vulnerability, its severity, impact,
            and remediation.
            """
        }
    ]
})

for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================


            The application allows users to set very weak passwords
            such as 123456 and does not enforce password strength.
            Explain the vulnerability, its severity, impact,
            and remediation.
            
================================== Ai Message ==================================
Tool Calls:
  get_vulnerability_guidance (fc_a7944384-42b8-4962-bc4a-827b93732373)
 Call ID: fc_a7944384-42b8-4962-bc4a-827b93732373
  Args:
    vulnerability: Weak Password Policy
================================= Tool Message =================================
Name: get_vulnerability_guidance


        Enforce strong password policies.
        Store passwords using a secure password hashing
        algorithm such as Argon2id or bcrypt.
        Enable rate limiting and MFA where appropriate.
        
================================== Ai Message ==================================
Tool Calls:

In [25]:
record = extractor.invoke(
    "The application allows users to set very weak passwords such as 123456."
)

print(record.model_dump())


{'vulnerability': 'Weak Password Policy', 'severity': 'High', 'impact': 'Allows attackers to easily guess or brute-force user credentials, leading to unauthorized access, data breach, and potential escalation of privileges.', 'remediation': 'Enforce a strong password policy requiring a minimum length (e.g., 12 characters), inclusion of uppercase, lowercase, numbers, and special characters. Implement password strength meters, disallow common passwords, and consider multi-factor authentication (MFA).', 'verification': "Attempt to create an account with a weak password (e.g., '123456') and verify that the system rejects it. Test password strength enforcement using a list of common passwords and ensure MFA prompts are triggered for privileged accounts.", 'confidence': 'High'}


In [26]:
save_result = log_vulnerability_assessment.invoke(
    record.model_dump()
)

print(save_result)

Vulnerability assessment saved successfully.
